In [1]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path.cwd().parents[1] / "data"
OUT = Path.cwd() / "results"

P_MIN = 1
OUTLET_DEFERRAL_WINDOW = 7
SUFFICIENT_WEIGHT_UTIL = 0.8
SUFFICIENT_VOLUME_UTIL = 0.8
prm = dict(P_MIN=P_MIN, OUTLET_DEFERRAL_WINDOW=OUTLET_DEFERRAL_WINDOW,
           SUFFICIENT_WEIGHT_UTIL=SUFFICIENT_WEIGHT_UTIL, SUFFICIENT_VOLUME_UTIL=SUFFICIENT_VOLUME_UTIL)

DECISIONS = Path.cwd() / "results/algo1_decisions.csv"
ROUTES = Path.cwd() / "results/algo1_routes.csv"
COMPARE_HISTORICAL = True
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

SCENARIO = "S1"
BUDGET = {"Fresh": 270, "Style": 480, "Tech": 480}

In [2]:
UNCONFIRMED, DISTRICT_BRAND_CONFIRMED, ROUTE_LEGS_CONFIRMED = 0, 1, 2
STATE_NAME = {UNCONFIRMED: "UNCONFIRMED", DISTRICT_BRAND_CONFIRMED: "DISTRICT_BRAND_CONFIRMED",
              ROUTE_LEGS_CONFIRMED: "ROUTE_LEGS_CONFIRMED"}
FRESH = {"fresh_chilled", "fresh_dry"}
EXACT_PACK_MAX = 12

In [3]:
@dataclass(eq=False)
class Order:
    id: str
    order_date: str
    outlet: str
    brand: str
    cat: str
    district: str
    depot: str
    w: float
    v: float
    van_only: bool
    times_deferred: int = 0
    p: int = 0

In [4]:
@dataclass(eq=False)
class Vehicle:
    id: str
    type: str
    temp: str
    depot: str
    W: float
    V: float
    weekly_km: float
    used_week: float = 0.0

In [5]:
@dataclass(eq=False)
class Trip:
    veh: Vehicle
    slot: int
    state: int = UNCONFIRMED
    district: str = None
    cat: str = None
    orders: list = field(default_factory=list)
    w: float = 0.0
    v: float = 0.0

In [6]:
class DayPlanner:
    def __init__(self, vehicles, travel, prm):
        self.travel, self.prm = travel, prm
        self.trips = [Trip(veh, slot) for veh in vehicles for slot in (1, 2)]
        self.by_vehicle = defaultdict(list)
        for t in self.trips:
            self.by_vehicle[t.veh.id].append(t)

    def route_km(self, district, outlets):
        if not outlets:
            return 0.0
        d = self.travel[district]
        return 2 * d["depot_to_district_km"] + d["inter_stop_km"] * (len(outlets) - 1)

    def trip_km(self, trip):
        return self.route_km(trip.district, {o.outlet for o in trip.orders})

    def km_left(self, veh, excluding=None):
        today = sum(self.trip_km(t) for t in self.by_vehicle[veh.id] if t is not excluding and t.orders)
        return veh.weekly_km - veh.used_week - today

    @staticmethod
    def compatible(trip, o):
        veh = trip.veh
        if o.depot != veh.depot:
            return False
        if o.cat == "fresh_chilled" and veh.temp != "reefer":
            return False
        if o.van_only and veh.type != "van":
            return False
        if trip.slot == 1 and o.cat not in FRESH:
            return False
        return True

    def fits(self, trip, orders):
        if trip.state == ROUTE_LEGS_CONFIRMED or not orders:
            return False
        district, cat = (trip.district, trip.cat) if trip.orders else (orders[0].district, orders[0].cat)
        for o in orders:
            if o.district != district or o.cat != cat or not self.compatible(trip, o):
                return False
        if trip.w + sum(o.w for o in orders) > trip.veh.W + 1e-9:
            return False
        if trip.v + sum(o.v for o in orders) > trip.veh.V + 1e-9:
            return False
        outlets = {o.outlet for o in trip.orders} | {o.outlet for o in orders}
        return self.route_km(district, outlets) <= self.km_left(trip.veh, excluding=trip) + 1e-9

    def assign(self, trip, orders, state):
        if not trip.orders:
            trip.district, trip.cat = orders[0].district, orders[0].cat
        trip.orders.extend(orders)
        trip.w += sum(o.w for o in orders)
        trip.v += sum(o.v for o in orders)
        trip.state = max(trip.state, state)

    @staticmethod
    def util(trip, orders=()):
        return (trip.w + sum(o.w for o in orders)) / trip.veh.W, (trip.v + sum(o.v for o in orders)) / trip.veh.V

    def free(self, depot, slots, temps=None, types=None):
        return [t for t in self.trips if t.state == UNCONFIRMED and not t.orders and t.veh.depot == depot
                and t.slot in slots and (temps is None or t.veh.temp in temps)
                and (types is None or t.veh.type in types)]

    def size_key(self, trip):
        return trip.veh.W, trip.veh.V, -self.km_left(trip.veh)

    def smallest_fitting(self, cands, orders):
        for t in sorted(cands, key=self.size_key):
            if self.fits(t, orders):
                return t
        return None

    def best_subset(self, trip, orders):
        if len(orders) <= EXACT_PACK_MAX:
            best, best_score = [], -1.0
            for r in range(len(orders), 0, -1):
                for combo in itertools.combinations(orders, r):
                    combo = list(combo)
                    if self.fits(trip, combo):
                        score = max(self.util(trip, combo))
                        if score > best_score + 1e-12:
                            best, best_score = combo, score
            return best
        chosen = []
        for o in sorted(orders, key=lambda o: -o.v):
            if self.fits(trip, chosen + [o]):
                chosen.append(o)
        return chosen

    def split_pick(self, cands, orders):
        sw, sv = self.prm["SUFFICIENT_WEIGHT_UTIL"], self.prm["SUFFICIENT_VOLUME_UTIL"]
        best = None
        for t in sorted(cands, key=self.size_key):
            chosen = []
            for o in sorted(orders, key=lambda o: -max(o.w / t.veh.W, o.v / t.veh.V)):
                if self.fits(t, chosen + [o]):
                    chosen.append(o)
                    uw, uv = self.util(t, chosen)
                    if uw >= sw or uv >= sv:
                        return t, chosen
            if chosen:
                score = max(self.util(t, chosen))
                if best is None or score > best[2]:
                    best = (t, chosen, score)
        return (best[0], best[1]) if best else None

    def top_up(self, orders, cats=None):
        left = list(orders)
        for t in self.trips:
            if t.state != DISTRICT_BRAND_CONFIRMED:
                continue
            if cats and t.cat not in cats:
                continue
            for o in sorted([o for o in left if o.district == t.district and o.cat == t.cat], key=lambda o: o.v):
                if self.fits(t, [o]):
                    self.assign(t, [o], DISTRICT_BRAND_CONFIRMED)
                    left.remove(o)
        return left

    def serve_van_only(self, orders, cand_groups):
        left = list(orders)
        for make in cand_groups:
            if not left:
                break
            cands = make()
            fitting = [t for t in cands if self.fits(t, left)]
            if fitting:
                t = max(fitting, key=lambda t: self.km_left(t.veh))
                self.assign(t, left, DISTRICT_BRAND_CONFIRMED)
                return []
            for t in sorted(cands, key=lambda t: -self.km_left(t.veh)):
                if not left:
                    break
                chosen = self.best_subset(t, left)
                if chosen:
                    self.assign(t, chosen, DISTRICT_BRAND_CONFIRMED)
                    left = [o for o in left if o not in chosen]
        return left

    def serve_district(self, orders, first_cands, more_cands):
        left = list(orders)
        t = self.smallest_fitting(first_cands(), left)
        if t:
            self.assign(t, left, DISTRICT_BRAND_CONFIRMED)
            return []
        make = first_cands
        while left:
            pick = self.split_pick(make(), left)
            if pick is None and make is first_cands:
                make = more_cands
                pick = self.split_pick(make(), left)
            if pick is None:
                return left
            t, chosen = pick
            self.assign(t, chosen, ROUTE_LEGS_CONFIRMED)
            left = [o for o in left if o not in chosen]
            make = more_cands
            t = self.smallest_fitting(make(), left) if left else None
            if t:
                self.assign(t, left, DISTRICT_BRAND_CONFIRMED)
                return []
        return []

    def step_van_only(self, orders, cat, cand_groups):
        mine = [o for o in orders if o.cat == cat and o.van_only]
        rest = [o for o in orders if not (o.cat == cat and o.van_only)]
        groups = defaultdict(list)
        for o in mine:
            groups[(o.depot, o.district)].append(o)
        for (depot, _), group in groups.items():
            rest += self.serve_van_only(group, [lambda g=g, d=depot: g(d) for g in cand_groups])
        return rest

    def step_trucks(self, orders, cat, first, more):
        rest = self.top_up(orders, cats={cat})
        mine = [o for o in rest if o.cat == cat and not o.van_only]
        rest = [o for o in rest if not (o.cat == cat and not o.van_only)]
        groups = defaultdict(list)
        for o in mine:
            groups[(o.depot, o.district)].append(o)
        for (depot, district) in sorted(groups, key=lambda k: -self.travel[k[1]]["depot_to_district_km"]):
            rest += self.serve_district(groups[(depot, district)], lambda d=depot: first(d), lambda d=depot: more(d))
        return rest

    def confirm(self, lower):
        sw, sv = self.prm["SUFFICIENT_WEIGHT_UTIL"], self.prm["SUFFICIENT_VOLUME_UTIL"]
        waiting = {(o.depot, o.district, o.cat) for o in lower}
        for t in self.trips:
            if t.state == DISTRICT_BRAND_CONFIRMED and (t.veh.depot, t.district, t.cat) not in waiting:
                uw, uv = self.util(t)
                if uw > sw and uv > sv:
                    t.state = ROUTE_LEGS_CONFIRMED

    def levels(self, orders):
        pmin = self.prm["P_MIN"]
        high = sorted({o.p for o in orders if o.p >= pmin}, reverse=True)
        out = [[o for o in orders if o.p == p] for p in high]
        low = [o for o in orders if o.p < pmin]
        if low:
            out.append(low)
        return out

    def plan(self, orders):
        f = self.free
        reefer_vans = lambda slot: (lambda d: f(d, {slot}, temps={"reefer"}, types={"van"}))
        amb_vans = lambda slot: (lambda d: f(d, {slot}, temps={"ambient"}, types={"van"}))
        deferred = []
        levels = self.levels(orders)
        for i, lvl in enumerate(levels):
            lower = [o for l in levels[i + 1:] for o in l]
            rest = self.top_up(lvl)
            rest = self.step_van_only(rest, "fresh_chilled", [reefer_vans(1), reefer_vans(2)])
            rest = self.step_trucks(rest, "fresh_chilled",
                                    first=lambda d: f(d, {1}, temps={"reefer"}),
                                    more=lambda d: f(d, {1, 2}, temps={"reefer"}))
            rest = self.step_van_only(rest, "fresh_dry",
                                      [amb_vans(1), lambda d: f(d, {2}, types={"van"})])
            rest = self.step_trucks(rest, "fresh_dry",
                                    first=lambda d: f(d, {1}, temps={"ambient"}) + f(d, {2}, temps={"reefer"}),
                                    more=lambda d: f(d, {1, 2}, temps={"ambient"}) + f(d, {2}, temps={"reefer"}))
            for cat in ("style", "tech"):
                rest = self.step_van_only(rest, cat, [lambda d: f(d, {2}, types={"van"})])
                rest = self.step_trucks(rest, cat, first=lambda d: f(d, {2}), more=lambda d: f(d, {2}))
            deferred += rest
            self.confirm(lower)
        for t in self.trips:
            if t.orders:
                t.state = ROUTE_LEGS_CONFIRMED
        return deferred

In [7]:
orders_df = pd.concat([pd.read_csv(DATA / "Training Data/deliveries_train.csv"),
                       pd.read_csv(DATA / "Test Data/task1_test_inputs.csv")], ignore_index=True)
outlets = pd.read_csv(DATA / "General Data/outlets.csv").set_index("outlet_id")
vehicles_df = pd.read_csv(DATA / "General Data/vehicles.csv").rename(columns=str.strip)
if "weekly_range_km" not in vehicles_df:
    vehicles_df["weekly_range_km"] = vehicles_df.km_per_l * vehicles_df.weekly_fuel_quota_l
travel = pd.read_csv(DATA / "General Data/district_travel.csv").set_index("district").to_dict("index")

van_only = outlets.parking_constraint.eq("van_only")
cat = orders_df.brand.str.lower().where(orders_df.brand != "Fresh",
                                        "fresh_" + orders_df.temp_requirement.map({"chilled": "chilled", "ambient": "dry"}))
orders = [Order(r.delivery_id, r.order_date, r.outlet_id, r.brand, c, r.district, r.depot,
                r.order_weight_kg, r.order_volume_m3, bool(van_only[r.outlet_id]))
          for r, c in zip(orders_df.itertuples(), cat)]
vehicles = [Vehicle(r.vehicle_id, r.type, r.temp, r.depot, r.weight_cap_kg, r.volume_cap_m3, r.weekly_range_km)
            for r in vehicles_df.itertuples()]
by_date = defaultdict(list)
for o in orders:
    by_date[o.order_date].append(o)
dates = sorted(by_date)
print(f"{len(dates)} days, {len(orders):,} orders, {len(vehicles)} vehicles; hyperparameters: {prm}")

695 days, 97,321 orders, 60 vehicles; hyperparameters: {'P_MIN': 1, 'OUTLET_DEFERRAL_WINDOW': 7, 'SUFFICIENT_WEIGHT_UTIL': 0.8, 'SUFFICIENT_VOLUME_UTIL': 0.8}


In [8]:
outlet_deferrals = defaultdict(deque)
window = pd.Timedelta(days=OUTLET_DEFERRAL_WINDOW)
pending, week, decisions, routes = [], None, [], []
for date in dates:
    day = pd.Timestamp(date)
    if day.isocalendar()[:2] != week:
        week = day.isocalendar()[:2]
        for v in vehicles:
            v.used_week = 0.0
    todays = pending + by_date[date]
    for o in todays:
        hist = outlet_deferrals[o.outlet]
        while hist and hist[0] < day - window:
            hist.popleft()
        o.p = o.times_deferred * len(hist)

    planner = DayPlanner(vehicles, travel, prm)
    deferred = planner.plan(todays)
    deferred_ids = {id(o) for o in deferred}

    where = {}
    for t in planner.trips:
        if not t.orders:
            continue
        route_id = f"{date}-{t.veh.id}-T{t.slot}"
        uw, uv = planner.util(t)
        routes.append(dict(plan_date=date, route_id=route_id, depot=t.veh.depot, vehicle_id=t.veh.id,
                           vehicle_type=t.veh.type, vehicle_temp=t.veh.temp, trip_id=t.slot,
                           district=t.district, category=t.cat, orders=len(t.orders),
                           stops=len({o.outlet for o in t.orders}), weight_kg=round(t.w, 1),
                           volume_m3=round(t.v, 3), weight_util=round(uw, 4), volume_util=round(uv, 4),
                           km=round(planner.trip_km(t), 1), final_state=STATE_NAME[t.state]))
        for o in t.orders:
            where[id(o)] = (t.veh.id, t.slot, route_id)
    for t in planner.trips:
        if t.orders:
            t.veh.used_week += planner.trip_km(t)

    for o in todays:
        veh, slot, route_id = where.get(id(o), ("", "", ""))
        decisions.append(dict(plan_date=date, delivery_id=o.id, order_date=o.order_date, outlet_id=o.outlet,
                              brand=o.brand, category=o.cat, district=o.district, depot=o.depot,
                              van_only=o.van_only, weight_kg=o.w, volume_m3=o.v,
                              times_deferred_before=o.times_deferred, p=o.p,
                              decision="deferred" if id(o) in deferred_ids else "served",
                              vehicle_id=veh, trip_id=slot, route_id=route_id))
    for o in deferred:
        o.times_deferred += 1
        outlet_deferrals[o.outlet].append(day)
    pending = deferred

dec = pd.DataFrame(decisions)
rts = pd.DataFrame(routes)
print(f"{len(dec):,} decisions ({(dec.decision == 'deferred').sum():,} deferrals), {len(rts):,} vehicle-trips; "
      f"{len(pending)} orders still pending at the end")

97,321 decisions (0 deferrals), 22,440 vehicle-trips; 0 orders still pending at the end


In [9]:
OUT.mkdir(exist_ok=True)
dec.to_csv(OUT / "algo1_decisions.csv", index=False)
rts.to_csv(OUT / "algo1_routes.csv", index=False)
rts.groupby("category")[["weight_util", "volume_util"]].median().round(2)

,weight_util,volume_util
category,,
fresh_chilled,0.32,0.34
fresh_dry,0.64,0.58
style,0.30,0.88
tech,0.58,0.30


In [10]:
dec = pd.read_csv(DECISIONS)
all_outlets = pd.read_csv(DATA / "General Data/outlets.csv").outlet_id
runs = {"algorithm": dec}

In [11]:
if COMPARE_HISTORICAL:
    h = pd.concat([pd.read_csv(DATA / "Training Data/deliveries_train.csv"),
                   pd.read_csv(DATA / "Test Data/task1_test_inputs.csv")], ignore_index=True)
    h["category"] = h.brand.str.lower().where(h.brand != "Fresh",
                                              "fresh_" + h.temp_requirement.map({"chilled": "chilled", "ambient": "dry"}))
    hist_days = sorted(h.order_date.unique())
    idx = {d: i for i, d in enumerate(hist_days)}
    h = h[h.delivery_id.isin(set(dec.delivery_id))]
    rows = []
    for r in h.itertuples():
        start = idx[r.order_date]
        if r.dispatch_status == "attempted":
            end = idx.get(r.dispatch_date, start)
        elif r.dispatch_status == "deferred":
            end = idx.get(r.dispatch_date, len(hist_days))
        else:
            end = start + 1
        base = dict(delivery_id=r.delivery_id, outlet_id=r.outlet_id, category=r.category, district=r.district)
        for i in range(start, min(end, len(hist_days))):
            rows.append(dict(base, plan_date=hist_days[i], decision="deferred"))
        if r.dispatch_status != "not_run" and end < len(hist_days):
            rows.append(dict(base, plan_date=hist_days[end], decision="served"))
    runs["historical"] = pd.DataFrame(rows)

In [12]:
summary, tables = {}, {}
for name, d in runs.items():
    orders = d.groupby("delivery_id").agg(outlet_id=("outlet_id", "first"), category=("category", "first"),
                                          district=("district", "first"),
                                          deferrals=("decision", lambda s: (s == "deferred").sum()),
                                          served=("decision", lambda s: (s == "served").any()))
    day_index = {x: i for i, x in enumerate(sorted(d.plan_date.unique()))}
    ev = d[d.decision == "deferred"][["delivery_id", "plan_date"]].copy()
    if ev.empty:
        orders["longest_run"] = 0
    else:
        ev["i"] = ev.plan_date.map(day_index)
        ev = ev.sort_values(["delivery_id", "i"])
        ev["run_id"] = ((ev.i.diff() != 1) | (ev.delivery_id != ev.delivery_id.shift())).cumsum()
        orders["longest_run"] = ev.groupby(["delivery_id", "run_id"]).size().groupby("delivery_id").max()
        orders["longest_run"] = orders.longest_run.fillna(0).astype(int)
    events = d[d.decision == "deferred"]
    per_outlet = events.groupby("outlet_id").size().reindex(all_outlets, fill_value=0)
    x = np.sort(per_outlet.values.astype(float))
    gini = float((2 * np.arange(1, len(x) + 1) - len(x) - 1).dot(x) / (len(x) * x.sum())) if x.sum() else 0.0
    deferred_orders = orders[orders.deferrals > 0]
    summary[name] = {
        "orders": len(orders),
        "deferral events (order-days)": len(events),
        "orders ever deferred": len(deferred_orders),
        "share of orders deferred": f"{len(deferred_orders) / len(orders):.1%}",
        "orders never served": int((~orders.served).sum()),
        "longest continuous deferral (days)": int(orders.longest_run.max()),
        "mean run for deferred orders (days)": round(deferred_orders.longest_run.mean(), 2) if len(deferred_orders) else 0,
        "orders deferred 2+ days in a row": int((orders.longest_run >= 2).sum()),
        "orders deferred 3+ days in a row": int((orders.longest_run >= 3).sum()),
        "outlets with any deferral": f"{int((per_outlet > 0).sum())} of {len(all_outlets)}",
        "deferrals at the worst outlet": int(per_outlet.max()),
        "share of deferrals at the worst 5 outlets": f"{per_outlet.nlargest(5).sum() / per_outlet.sum():.1%}"
        if per_outlet.sum() else "0.0%",
        "Gini of deferrals across outlets": round(gini, 2),
    }
    run_dist = deferred_orders.longest_run.clip(upper=5).value_counts().sort_index()
    run_dist.index = [f"{i}+" if i == 5 else str(i) for i in run_dist.index]
    tables[name] = {
        "by category": events.groupby("category").size(),
        "by district": events.groupby("district").size(),
        "by weekday": events.groupby(pd.to_datetime(events.plan_date).dt.dayofweek).size().rename(index=dict(enumerate(DAYS))),
        "by longest continuous run (days)": run_dist,
        "worst 10 outlets": per_outlet[per_outlet > 0].nlargest(10),
    }
pd.DataFrame(summary)

,algorithm,historical
orders,97321,97321
deferral events (order-days),0,2430
orders ever deferred,0,2046
share of orders deferred,0.0%,2.1%
orders never served,0,413
longest continuous deferral (days),0,2
mean run for deferred orders (days),0,1.19
orders deferred 2+ days in a row,0,384
orders deferred 3+ days in a row,0,0
outlets with any deferral,0 of 120,58 of 120


In [13]:
for title in tables["algorithm"]:
    t = pd.concat({n: tables[n][title] for n in runs}, axis=1).fillna(0).astype(int)
    print(f"\nDeferral events {title}\n{t.to_string() if len(t) else 'none'}")


Deferral events by category
               algorithm  historical
category                            
fresh_chilled          0        2402
fresh_dry              0          13
style                  0           7
tech                   0           8

Deferral events by district
              algorithm  historical
district                           
Badulla               0          15
Colombo               0        2056
Gampaha               0         316
Kalutara              0           7
Kandy                 0          16
Kegalle               0           3
Matale                0          12
Nuwara Eliya          0           5

Deferral events by weekday
           algorithm  historical
plan_date                       
Mon                0         375
Tue                0        1134
Wed                0          89
Thu                0         246
Fri                0         226
Sat                0         360

Deferral events by longest continuous run (days)
   algorithm  hist

In [14]:
if ROUTES.exists():
    rts = pd.read_csv(ROUTES)
    per_day = rts.groupby("plan_date")
    print(pd.Series({
        "vehicle-trips per day": len(rts) / rts.plan_date.nunique(),
        "vehicles used per day": per_day.vehicle_id.nunique().mean(),
        "vehicles running 2 trips per day": (rts.groupby(["plan_date", "vehicle_id"]).size() == 2).groupby("plan_date").sum().mean(),
        "stops per trip": rts.stops.mean(),
        "km per day": per_day.km.sum().mean(),
    }).round(2).to_string())
    display_fill = rts.groupby("category")[["weight_util", "volume_util"]].median().round(2)
    print(display_fill.to_string())

vehicle-trips per day                 32.29
vehicles used per day                 25.28
vehicles running 2 trips per day       7.01
stops per trip                         4.34
km per day                          4902.80
               weight_util  volume_util
category                               
fresh_chilled         0.32         0.34
fresh_dry             0.64         0.58
style                 0.30         0.88
tech                  0.58         0.30


In [15]:
s = pd.read_csv(DATA / "Test Data/task2b_peak_day_scenarios.csv")
s = s[s.scenario == SCENARIO].reset_index(drop=True)
fleet = pd.read_csv(DATA / "Test Data/task2b_peak_day_fleet.csv")
vehicles = pd.read_csv(DATA / "General Data/vehicles.csv").rename(columns=str.strip)
if "weekly_range_km" not in vehicles:
    vehicles["weekly_range_km"] = vehicles.km_per_l * vehicles.weekly_fuel_quota_l
vehicles = vehicles.merge(fleet[fleet.scenario == SCENARIO], on="vehicle_id")
travel = pd.read_csv(DATA / "General Data/district_travel.csv").set_index("district")
service = pd.read_csv(DATA / "General Data/service_allowance.csv").set_index(["brand", "dock_type"])
avail = vehicles[vehicles.status == "available"]
print(f"{len(s)} orders; {len(avail)} of {len(vehicles)} vehicles available")
avail[avail.temp == "reefer"][["vehicle_id", "type", "weight_cap_kg", "volume_cap_m3"]]

85 orders; 28 of 38 vehicles available


,vehicle_id,type,weight_cap_kg,volume_cap_m3
2,VEH003,truck,5510,26.4
5,VEH006,truck,6840,33.4
6,VEH007,truck,3610,19.4
35,VEH036,van,1040,7.0


In [16]:
orders = []
for r in s.itertuples():
    cat = r.brand.lower() if r.brand != "Fresh" else ("fresh_chilled" if r.temp_requirement == "chilled" else "fresh_dry")
    o = Order(r.order_ref, "scenario", r.outlet_id, r.brand, cat, r.district, r.depot, r.order_weight_kg,
              r.order_volume_m3, r.parking_constraint == "van_only", times_deferred=int(r.deferred_yesterday))
    o.p = int(r.deferred_yesterday) * int(r.days_since_last_served)
    orders.append(o)
vehs = [Vehicle(r.vehicle_id, r.type, r.temp, r.depot, r.weight_cap_kg, r.volume_cap_m3, r.weekly_range_km)
        for r in avail.itertuples()]

planner = DayPlanner(vehs, travel.to_dict("index"), prm)
deferred = {o.id for o in planner.plan(orders)}

used = defaultdict(list)
for t in planner.trips:
    if t.orders:
        used[t.veh.id].append(t)
placed = {o.id: (veh, n) for veh, ts in used.items()
          for n, t in enumerate(sorted(ts, key=lambda t: t.slot), start=1) for o in t.orders}
sub = pd.DataFrame([dict(scenario=SCENARIO, order_ref=r.order_ref, outlet_id=r.outlet_id,
                         decision="deferred" if r.order_ref in deferred else "served",
                         vehicle_id=placed.get(r.order_ref, ("", ""))[0],
                         trip_id=placed.get(r.order_ref, ("", ""))[1]) for r in s.itertuples()])
OUT.mkdir(exist_ok=True)
sub.to_csv(OUT / "task2b_algo1_submission.csv", index=False)
print(f"served {(sub.decision == 'served').sum()} of {len(sub)}")

served 70 of 85


In [17]:
v_idx = vehicles.set_index("vehicle_id")
served = sub[sub.decision == "served"].merge(
    s[["order_ref", "brand", "district", "depot", "dock_type", "parking_constraint", "temp_requirement",
       "order_weight_kg", "order_volume_m3"]], on="order_ref")
violations, trip_rows = [], []
if served.order_ref.duplicated().any():
    violations.append("rule 5: an order is assigned more than once")
for (veh, trip), rows in served.groupby(["vehicle_id", "trip_id"]):
    car, tag = v_idx.loc[veh], f"{veh} trip {trip}"
    if car.status != "available":
        violations.append(f"{tag}: vehicle is {car.status}")
    if rows.brand.nunique() > 1 or rows.district.nunique() > 1:
        violations.append(f"rule 1: {tag} mixes brands or districts")
    if (rows.temp_requirement == "chilled").any() and car.temp != "reefer":
        violations.append(f"rule 2: {tag} carries chilled orders on an ambient vehicle")
    if (rows.parking_constraint == "van_only").any() and car.type != "van":
        violations.append(f"rule 3: {tag} serves a van-only outlet with a {car.type}")
    if (rows.depot != car.depot).any():
        violations.append(f"rule 4: {tag} serves another depot's outlets")
    w, vol = rows.order_weight_kg.sum(), rows.order_volume_m3.sum()
    if w > car.weight_cap_kg + 1e-9 or vol > car.volume_cap_m3 + 1e-9:
        violations.append(f"rule 6: {tag} over capacity ({w:,.0f}/{car.weight_cap_kg:,} kg, {vol:.2f}/{car.volume_cap_m3} m³)")
    d = travel.loc[rows.district.iloc[0]]
    minutes = (d.depot_to_district_freeflow_min + d.inter_stop_freeflow_min * (len(rows) - 1)
               + sum(service.loc[(b, dk), "service_allowance_min"] for b, dk in zip(rows.brand, rows.dock_type)))
    trip_rows.append(dict(vehicle_id=veh, trip_id=trip, vehicle_type=car.type, vehicle_temp=car.temp,
                          brand=rows.brand.iloc[0], district=rows.district.iloc[0], orders=len(rows),
                          chilled_orders=int((rows.temp_requirement == "chilled").sum()),
                          weight_util=round(w / car.weight_cap_kg, 3), volume_util=round(vol / car.volume_cap_m3, 3),
                          minutes=int(minutes)))
trips = pd.DataFrame(trip_rows).sort_values(["vehicle_id", "trip_id"])
for veh, rows in trips.groupby("vehicle_id"):
    if len(rows) > 2 or sorted(rows.trip_id) not in ([1], [1, 2]):
        violations.append(f"rule 7: {veh} has trips {sorted(rows.trip_id)}")
    fresh_min = rows.loc[rows.brand == "Fresh", "minutes"].sum()
    other_min = rows.loc[rows.brand != "Fresh", "minutes"].sum()
    if fresh_min > BUDGET["Fresh"]:
        violations.append(f"rule 7: {veh} Fresh trips take {fresh_min} min (budget 270)")
    if other_min > BUDGET["Style"]:
        violations.append(f"rule 7: {veh} Style/Tech trips take {other_min} min (budget 480)")
print("All booklet rules pass." if not violations else f"{len(violations)} violations:\n" + "\n".join(violations))
trips

2 violations:
rule 7: VEH006 Fresh trips take 387 min (budget 270)
rule 7: VEH018 Fresh trips take 278 min (budget 270)


,vehicle_id,trip_id,vehicle_type,vehicle_temp,brand,district,orders,chilled_orders,weight_util,volume_util,minutes
0,VEH003,1,truck,reefer,Fresh,Gampaha,2,2,0.743,0.801,76
1,VEH003,2,truck,reefer,Fresh,Galle,2,2,0.619,0.768,143
2,VEH006,1,truck,reefer,Fresh,Matara,2,2,0.323,0.359,177
3,VEH006,2,truck,reefer,Fresh,Kurunegala,3,3,0.672,0.750,210
4,VEH007,1,truck,reefer,Fresh,Puttalam,1,1,0.440,0.446,188
5,VEH007,2,truck,reefer,Style,Kurunegala,1,0,0.147,0.483,173
6,VEH008,1,truck,ambient,Fresh,Galle,6,0,0.719,0.695,242
7,VEH015,1,truck,ambient,Fresh,Matara,4,0,0.570,0.516,227
8,VEH015,2,truck,ambient,Style,Galle,2,0,0.346,0.895,217
9,VEH017,1,truck,ambient,Fresh,Puttalam,3,0,0.335,0.312,266


In [18]:
s = s.assign(decision=sub.decision, category=[o.cat for o in orders], p=[o.p for o in orders])
by_cat = s.groupby("category").agg(orders=("order_ref", "size"), served=("decision", lambda x: (x == "served").sum()),
                                   weight_kg=("order_weight_kg", "sum"), volume_m3=("order_volume_m3", "sum"))
by_cat["deferred"] = by_cat.orders - by_cat.served
by_cat.round(1)

,orders,served,weight_kg,volume_m3,deferred
category,,,,,
fresh_chilled,26,12,32780.4,181.6,14
fresh_dry,49,49,24838.4,133.0,0
style,5,4,4878.3,77.2,1
tech,5,5,5642.3,18.1,0


In [19]:
s.loc[s.decision == "deferred", ["order_ref", "outlet_id", "category", "district", "parking_constraint",
                                 "order_weight_kg", "order_volume_m3", "p"]].sort_values(["category", "district"])

,order_ref,outlet_id,category,district,parking_constraint,order_weight_kg,order_volume_m3,p
5,S1-005,OUT003,fresh_chilled,Colombo,van_only,318.1,1.725,0
7,S1-007,OUT004,fresh_chilled,Colombo,normal,567.7,2.925,0
9,S1-009,OUT005,fresh_chilled,Colombo,normal,1696.6,10.090,0
12,S1-012,OUT007,fresh_chilled,Colombo,normal,1991.0,11.722,0
14,S1-014,OUT008,fresh_chilled,Colombo,normal,713.9,4.003,0
16,S1-016,OUT009,fresh_chilled,Colombo,normal,1241.3,6.944,0
21,S1-021,OUT013,fresh_chilled,Colombo,normal,2007.0,9.898,0
28,S1-028,OUT026,fresh_chilled,Gampaha,normal,1190.1,6.368,0
31,S1-031,OUT028,fresh_chilled,Gampaha,normal,1209.9,6.883,0
33,S1-033,OUT029,fresh_chilled,Gampaha,normal,1036.8,6.227,0
